In [0]:
dbutils.library.restartPython()

In [0]:
import os
from  pyspark.sql.functions import *
from pyspark.sql.types import *
import sys


In [0]:
import sys, os

base = "/Workspace/Shared/fuel-prices-lakehouse/src/carburants"
sys.path.append(os.path.abspath(f"{base}/landing"))

from file_downloader import download_csv_file, read_csv_file


In [0]:
# url ="https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/prix-des-carburants-en-france-flux-instantane-v2/exports/csv?use_labels=true"
# path ="/Volumes/kyc/landing/data/carburants"

In [0]:
# file = download_csv_file(url, path)

In [0]:
file="/Volumes/kyc/landing/data/carburants/prix-des-carburants-en-france-flux-instantane-v2.csv"
carburants_df = read_csv_file(file, spark)

In [0]:
carburants_df.display()

In [0]:
carburants_df.printSchema()

In [0]:

schema_services = StructType([
    StructField("service", ArrayType(StringType()), True)
])

schema_prix = ArrayType(StructType([
    StructField("@nom", StringType(), True),
    StructField("@id", StringType(), True),
    StructField("@maj", StringType(), True),
    StructField("@valeur", StringType(), True),
]))

def build_fait_prix(df):
    df = df.withColumn("prix_parsed", from_json(col("prix"), schema_prix))
    df = df.withColumn("carburant", explode(col("prix_parsed")))
    return df.select(
        col("id").alias("station_id"),
        col("carburant.@nom").alias("nom_carburant"),
        col("carburant.@id").cast("int").alias("carburant_id"),
        col("carburant.@maj").cast(TimestampType()).alias("date_maj"),
        col("carburant.@valeur").cast(DoubleType()).alias("prix"),
    )

def build_fait_rupture(df):
    pass

def build_dim_service(df):
    df = df.withColumn("services_parsed", from_json(col("services"), schema_services))
    df = df.withColumn("service", explode(col("services_parsed.service")))
    return df.select(
        col("id").alias("station_id"),
        "service",
    )

def build_dim_geo(df):
    pass

def build_dim_station(df):
    return df.select(
        col("id").alias("station_id"),
        "adresse",
        "ville",
        "code_postal",
        "latitude",
        "longitude"
        "service"
    )

def build_carburant(df):
    pass

In [0]:
fait_prix = build_fait_prix(carburants_df, schema_prix)
fait_prix.display()